```
CONFIGURATION MINIMALE RECOMMANDEE
Python  3.13.x ou 3.12.x
PySpark 4.x
pandas  2.2.3
Java    21.x ou 25.x
```


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pathlib import Path

In [3]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DomesticFlightDelays")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)

In [4]:
project_path = Path.cwd()
flight_folder_path = project_path / "Stats_US_Flights_2019_2026"
passenger_folder_path = project_path / "T_T100D_MARKET_US_CARRIER_ONLY"
runways_folder_path = project_path / "Runways"

In [5]:
flight_files = [
    file.as_posix()
    for file in flight_folder_path.glob("*.csv")
]

passenger_files = [
    file.as_posix()
    for file in passenger_folder_path.glob("*.csv")
]

runways_files = [
    file.as_posix()
    for file in runways_folder_path.glob("*.csv")
]


In [6]:
COLUMNS_TO_KEEP_FLIGHTS = [
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "DOT_ID_Reporting_Airline",
    "OriginAirportID",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",
    "DestAirportID",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "Cancelled",
    "CancellationCode",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",
    "DistanceGroup",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "DivAirportLandings",
    "DivReachedDest",
    "DivActualElapsedTime",
    "DivArrDelay",
    "DivDistance"
]


FLIGHT_SCHEMA = StructType([

    StructField("Year", IntegerType(), True),
    StructField("Quarter", IntegerType(), True),
    StructField("Month", IntegerType(), True),
    StructField("DayofMonth", IntegerType(), True),
    StructField("DayOfWeek", IntegerType(), True),
    StructField("FlightDate", DateType(), True),

    StructField("Reporting_Airline", StringType(), True),
    StructField("DOT_ID_Reporting_Airline", IntegerType(), True),

    StructField("OriginAirportID", IntegerType(), True),
    StructField("Origin", StringType(), True),
    StructField("OriginCityName", StringType(), True),
    StructField("OriginState", StringType(), True),
    StructField("OriginStateName", StringType(), True),

    StructField("DestAirportID", IntegerType(), True),
    StructField("Dest", StringType(), True),
    StructField("DestCityName", StringType(), True),
    StructField("DestState", StringType(), True),
    StructField("DestStateName", StringType(), True),

    StructField("CRSDepTime", IntegerType(), True),
    StructField("DepTime", IntegerType(), True),
    StructField("DepDelay", DoubleType(), True),

    StructField("TaxiOut", DoubleType(), True),
    StructField("WheelsOff", IntegerType(), True),
    StructField("WheelsOn", IntegerType(), True),
    StructField("TaxiIn", DoubleType(), True),

    StructField("CRSArrTime", IntegerType(), True),
    StructField("ArrTime", IntegerType(), True),
    StructField("ArrDelay", DoubleType(), True),

    StructField("Cancelled", DoubleType(), True),
    StructField("CancellationCode", StringType(), True),

    StructField("CRSElapsedTime", DoubleType(), True),
    StructField("ActualElapsedTime", DoubleType(), True),
    StructField("AirTime", DoubleType(), True),

    StructField("Distance", DoubleType(), True),
    StructField("DistanceGroup", DoubleType(), True),

    StructField("CarrierDelay", DoubleType(), True),
    StructField("WeatherDelay", DoubleType(), True),
    StructField("NASDelay", DoubleType(), True),
    StructField("SecurityDelay", DoubleType(), True),
    StructField("LateAircraftDelay", DoubleType(), True),

    StructField("DivAirportLandings", IntegerType(), True),
    StructField("DivReachedDest", DoubleType(), True),
    StructField("DivActualElapsedTime", DoubleType(), True),
    StructField("DivArrDelay", DoubleType(), True),
    StructField("DivDistance", DoubleType(), True)
])

In [7]:
FLIGHT_COLUMN_MAPPING = {
    "Year": "YEAR",
    "Quarter": "QUARTER",
    "Month": "MONTH",
    "DayofMonth": "DAY_OF_MONTH",
    "DayOfWeek": "DAY_OF_WEEK",
    "FlightDate": "FLIGHT_DATE",

    "Reporting_Airline": "REPORTING_AIRLINE",
    "DOT_ID_Reporting_Airline": "DOT_ID_REPORTING_AIRLINE",

    "OriginAirportID": "ORIGIN_AIRPORT_ID",
    "Origin": "ORIGIN",
    "OriginCityName": "ORIGIN_CITY_NAME",
    "OriginState": "ORIGIN_STATE",
    "OriginStateName": "ORIGIN_STATE_NAME",

    "DestAirportID": "DEST_AIRPORT_ID",
    "Dest": "DEST",
    "DestCityName": "DEST_CITY_NAME",
    "DestState": "DEST_STATE",
    "DestStateName": "DEST_STATE_NAME",

    "CRSDepTime": "CRS_DEP_TIME",
    "DepTime": "DEP_TIME",
    "DepDelay": "DEP_DELAY",

    "TaxiOut": "TAXI_OUT",
    "WheelsOff": "WHEEL_OFF",
    "WheelsOn": "WHEEL_ON",
    "TaxiIn": "TAXI_IN",

    "CRSArrTime": "CRS_ARR_TIME",
    "ArrTime": "ARR_TIME",
    "ArrDelay": "ARR_DELAY",

    "Cancelled": "CANCELLED",
    "CancellationCode": "CANCELLATION_CODE",

    "CRSElapsedTime": "CRS_ELAPSED_TIME",
    "ActualElapsedTime": "ACTUAL_ELAPSED_TIME",
    "AirTime": "AIR_TIME",

    "Distance": "DISTANCE",
    "DistanceGroup": "DISTANCE_GROUP",

    "CarrierDelay": "CARRIER_DELAY",
    "WeatherDelay": "WEATHER_DELAY",
    "NASDelay": "NAS_DELAY",
    "SecurityDelay": "SECURITY_DELAY",
    "LateAircraftDelay": "LATE_AIRCRAFT_DELAY",

    "DivAirportLandings": "DIV_AIRPORT_LANDINGS",
    "DivReachedDest": "DIV_REACHED_DEST",
    "DivActualElapsedTime": "DIV_ACTUAL_ELAPSED_TIME",
    "DivArrDelay": "DIV_ARR_DELAY",
    "DivDistance": "DIV_DISTANCE"
}

In [8]:
df_flights = (spark.read.csv(
        flight_files,
        header=True,
        inferSchema=False
    )
    .select(COLUMNS_TO_KEEP_FLIGHTS)
)

In [9]:
df_flights = df_flights.select(* [F.col(field.name).cast(field.dataType).alias(FLIGHT_COLUMN_MAPPING[field.name])for field in FLIGHT_SCHEMA.fields])

In [8]:
COLUMNS_TO_KEEP_PASSENGERS = [
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED",
    "PAYLOAD",
    "SEATS",
    "PASSENGERS",
    "FREIGHT",
    "MAIL",
    "DISTANCE",
    "AIR_TIME",
    "UNIQUE_CARRIER",
    "AIRLINE_ID",
    "CARRIER_NAME",
    "ORIGIN_AIRPORT_ID",
    "ORIGIN",
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_ABR",
    "ORIGIN_STATE_NM",
    "DEST_AIRPORT_ID",
    "DEST",
    "DEST_CITY_NAME",
    "DEST_STATE_ABR",
    "DEST_STATE_NM",
    "YEAR",
    "QUARTER",
    "MONTH",
    "DISTANCE_GROUP"
]

PASSENGERS_SCHEMA = StructType([
    StructField("DEPARTURES_SCHEDULED", DoubleType(), True),
    StructField("DEPARTURES_PERFORMED", DoubleType(), True),
    StructField("PAYLOAD", DoubleType(), True),
    StructField("SEATS", DoubleType(), True),
    StructField("PASSENGERS", DoubleType(), True),
    StructField("FREIGHT", DoubleType(), True),
    StructField("MAIL", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("UNIQUE_CARRIER", StringType(), True),
    StructField("AIRLINE_ID", IntegerType(), True),
    StructField("CARRIER_NAME", StringType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("ORIGIN_STATE_NM", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("DEST_STATE_NM", StringType(), True),
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DISTANCE_GROUP", IntegerType(), True)
])

In [ ]:
df_passengers = (spark.read.csv(
        passenger_files,
        header=True,
        inferSchema=False
    ).select(COLUMNS_TO_KEEP_PASSENGERS)
)

In [153]:
for field in PASSENGERS_SCHEMA.fields:
    df_passengers = df_passengers.withColumn(
        field.name,
        F.col(field.name).cast(field.dataType)
    )

In [154]:
df_passengers.show()

+--------------------+--------------------+-------+-----+----------+-------+----+--------+--------+--------------+----------+--------------------+-----------------+------+-------------------+----------------+---------------+---------------+----+--------------------+--------------+-------------+----+-------+-----+--------------+
|DEPARTURES_SCHEDULED|DEPARTURES_PERFORMED|PAYLOAD|SEATS|PASSENGERS|FREIGHT|MAIL|DISTANCE|AIR_TIME|UNIQUE_CARRIER|AIRLINE_ID|        CARRIER_NAME|ORIGIN_AIRPORT_ID|ORIGIN|   ORIGIN_CITY_NAME|ORIGIN_STATE_ABR|ORIGIN_STATE_NM|DEST_AIRPORT_ID|DEST|      DEST_CITY_NAME|DEST_STATE_ABR|DEST_STATE_NM|YEAR|QUARTER|MONTH|DISTANCE_GROUP|
+--------------------+--------------------+-------+-----+----------+-------+----+--------+--------+--------------+----------+--------------------+-----------------+------+-------------------+----------------+---------------+---------------+----+--------------------+--------------+-------------+----+-------+-----+--------------+
|         

In [10]:
COLUMNS_TO_KEEP_AIRPORTS = [
    "OID",
    "STATE_CODE",
    "ARPT_ID",
    "ARPT_NAME",
    "CITY",
    "COUNTRY_CODE",
    "LAT1_DECIMAL",
    "LONG1_DECIMAL"
]

In [11]:
df_runways = spark.read.csv(runways_files, inferSchema= True, header= True).select(COLUMNS_TO_KEEP_AIRPORTS)

In [12]:
df_runways.show()

+---+----------+-------+--------------------+-----------------+------------+------------+-------------+
|OID|STATE_CODE|ARPT_ID|           ARPT_NAME|             CITY|COUNTRY_CODE|LAT1_DECIMAL|LONG1_DECIMAL|
+---+----------+-------+--------------------+-----------------+------------+------------+-------------+
|  1|        AL|    0J0|      ABBEVILLE MUNI|        ABBEVILLE|          US| 31.60852772| -85.23964763|
|  2|        AL|    2A8|             ADDISON|          ADDISON|          US|    34.21475|    -87.16145|
|  3|        AL|    EET|       SHELBY COUNTY|        ALABASTER|          US| 33.18411386| -86.78637688|
|  4|        AL|    BFZ|ALBERTVILLE RGNL/...|      ALBERTVILLE|          US| 34.22352341| -86.26330961|
|  5|        AL|    ALX|THOMAS C RUSSELL FLD|   ALEXANDER CITY|          US| 32.92218647|  -85.9633335|
|  6|        AL|    AIV|       GEORGE DOWNER|       ALICEVILLE|          US| 33.10348055| -88.20436111|
|  7|        AL|    79J|SOUTH ALABAMA RGN...|        ANDALUSIA| 

In [13]:
df_flights = df_flights.drop_duplicates()

In [14]:
first_condition = F.col("DEP_DELAY").isNull() | F.col("ARR_DELAY").isNull()
second_condition = F.col("CRS_ELAPSED_TIME").isNull() | F.col("ACTUAL_ELAPSED_TIME").isNull()
third_condition = F.col("CANCELLED") == 1.00

df_flights = df_flights.filter(third_condition | ~(first_condition & second_condition))

In [15]:
df_runways = df_runways.drop_duplicates(["ARPT_ID"])
rename_map = {
    "LAT1_DECIMAL": "LATITUDE",
    "LONG1_DECIMAL": "LONGITUDE"
}
df_runways = df_runways.withColumnsRenamed(rename_map)

In [16]:
df_flights = df_flights.join(df_runways, (df_flights["ORIGIN"] == df_runways["ARPT_ID"]) & (df_flights["ORIGIN_STATE"] == df_runways["STATE_CODE"]),
                              "inner").select(df_flights["*"], df_runways["LATITUDE"], df_runways["LONGITUDE"])

In [15]:
df_passengers = df_passengers.filter(F.col("PASSENGERS") != 0.0).sort("PASSENGERS")

NameError: name 'df_passengers' is not defined

In [ ]:
flight_silver_path = project_path / "Data" / "Silver" / "Flights"

(df_flights.write
    .mode("overwrite")
    .partitionBy("YEAR", "MONTH")
    .parquet(flight_silver_path.as_posix())
)

In [16]:
df_flights.show(10)

+----+-------+-----+------------+-----------+-----------+-----------------+------------------------+-----------------+------+----------------+------------+-----------------+---------------+----+--------------------+----------+---------------+------------+--------+---------+--------+---------+--------+-------+------------+--------+---------+---------+-----------------+----------------+-------------------+--------+--------+--------------+-------------+-------------+---------+--------------+-------------------+--------------------+----------------+-----------------------+-------------+------------+-----------+-------------+
|YEAR|QUARTER|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|FLIGHT_DATE|REPORTING_AIRLINE|DOT_ID_REPORTING_AIRLINE|ORIGIN_AIRPORT_ID|ORIGIN|ORIGIN_CITY_NAME|ORIGIN_STATE|ORIGIN_STATE_NAME|DEST_AIRPORT_ID|DEST|      DEST_CITY_NAME|DEST_STATE|DEST_STATE_NAME|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEEL_OFF|WHEEL_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|C

In [ ]:
df_passengers.show(10)